In [1]:
import json
import datetime
from pathlib import Path

import joblib
import pandas as pd
import sklearn

# ---- Settings (change only these) ----
OUT_DIR = Path("../outputs")
MODEL_FILE = OUT_DIR / "best_churn_model.joblib"
METRICS_FILE = OUT_DIR / "model_metrics.csv"
CLEAN_FILE = OUT_DIR / "customers_cleaned.csv"    # optional, used for dataset info
THRESHOLD = 0.5
VERSION = "v1"
# --------------------------------------

# 1. Load the saved model bundle (contains preprocess, model, name)
bundle = joblib.load(MODEL_FILE)
model_name = bundle["name"]

# 2. Read the saved metrics and pick this model's row
metrics = pd.read_csv(METRICS_FILE).set_index("Model")
if model_name not in metrics.index:
    raise ValueError(f"'{model_name}' not found in {METRICS_FILE}. Found: {list(metrics.index)}")
test_metrics = metrics.loc[model_name].round(3).to_dict()

# 3. Dataset info (optional, skipped if the file is missing)
dataset = {"file": CLEAN_FILE.name}
if CLEAN_FILE.exists():
    d = pd.read_csv(CLEAN_FILE)
    dataset.update(rows=len(d), columns=d.shape[1],
                   churn_rate_pct=round((d["Churn"] == "Yes").mean() * 100, 2))

# 4. Build and save the model card
card = {
    "model": model_name,
    "model_class": type(bundle["model"]).__name__,
    "version": VERSION,
    "trained_on": dataset,
    "card_created": datetime.date.today().isoformat(),
    "library_versions": {"scikit-learn": sklearn.__version__,
                         "pandas": pd.__version__,
                         "joblib": joblib.__version__},
    "test_metrics": test_metrics,
    "threshold": THRESHOLD,
    "input_features": list(bundle["preprocess"].feature_names_in_),
}

OUT_DIR.mkdir(exist_ok=True)
with open(OUT_DIR / "model_card.json", "w") as f:
    json.dump(card, f, indent=2)

print(json.dumps(card, indent=2))

{
  "model": "Logistic Regression",
  "model_class": "LogisticRegression",
  "version": "v1",
  "trained_on": {
    "file": "customers_cleaned.csv",
    "rows": 7043,
    "columns": 21,
    "churn_rate_pct": 26.54
  },
  "card_created": "2026-10-12",
  "library_versions": {
    "scikit-learn": "1.6.1",
    "pandas": "2.2.3",
    "joblib": "1.5.0"
  },
  "test_metrics": {
    "Accuracy": 0.738,
    "Precision": 0.504,
    "Recall": 0.783,
    "F1-score": 0.614,
    "ROC-AUC": 0.842
  },
  "threshold": 0.5,
  "input_features": [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges"
  ]
}
